<a href="https://colab.research.google.com/github/Aman6665/NLP-Program-using-Python-NLTK-and-spaCy/blob/main/26_Text_Generation_using_GPT_2_on_a_custom_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning GPT-2 for Text Generation on a Custom Dataset

This notebook demonstrates how to fine-tune a pre-trained GPT-2 model on a custom text dataset using the Hugging Face `transformers` and `accelerate` libraries.

In [1]:
!pip install -q transformers datasets accelerate evaluate

## 1. Prepare a Custom Dataset
For demonstration, we will write a sample custom text file. In a real-world scenario, you can upload your own `.txt` or `.csv` files.

In [2]:
custom_text = """The history of artificial intelligence (AI) began in antiquity, with myths, stories and rumors of artificial beings endowed with intelligence or consciousness by master craftsmen.
As computers became more powerful, AI research emerged as an academic discipline in the mid-20th century.
Generative models like GPT-2 are designed to predict the next word in a sequence given all previous words.
By training on large amounts of text, these models learn grammar, facts, and reasoning capabilities.
Fine-tuning allows us to adapt these general models to specific styles, topics, or tasks.
"""

with open('custom_dataset.txt', 'w') as f:
    f.write(custom_text)

print("Custom dataset saved to 'custom_dataset.txt'")

Custom dataset saved to 'custom_dataset.txt'


## 2. Load the Model, Tokenizer, and Dataset

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import load_dataset

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
# GPT-2 does not have a padding token by default, so we define one
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))

# Load our local custom text file
dataset = load_dataset("text", data_files={"train": "custom_dataset.txt"})

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

## 3. Preprocess and Tokenize the Dataset

In [4]:
def tokenize_function(examples):
    outputs = tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")
    # For causal LM, targets are the same as input IDs shifted
    outputs["labels"] = outputs["input_ids"].copy()
    return outputs

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print(tokenized_dataset)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5
    })
})


## 4. Set Up Training Arguments and Fine-Tune

In [7]:
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    save_steps=100,
    logging_steps=10,
    learning_rate=5e-5,
    prediction_loss_only=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
)

trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,2.686652


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15, training_loss=1.9993782043457031, metrics={'train_runtime': 52.0701, 'train_samples_per_second': 0.48, 'train_steps_per_second': 0.288, 'total_flos': 1633075200000.0, 'train_loss': 1.9993782043457031, 'epoch': 5.0})

## 5. Generate Text with the Fine-Tuned Model

In [6]:
prompt = "Generative models"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

# Generate text
output_sequences = model.generate(
    input_ids=input_ids,
    max_length=50,
    num_return_sequences=1,
    no_repeat_ngram_size=2,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id
)

generated_text = tokenizer.decode(output_sequences[0], skip_special_tokens=True)
print("Generated text:")
print(generated_text)

Generated text:
Generative models of the future:

In the 21st century, we will be watching the rapid evolution of our culture, which is based on a culture of intellectual and emotional maturity. We will have an abundance of information about the world. The
